# 실험① — 기준선과 교차검증 전제

**이 노트북이 답할 질문**

> 이미지에서 값을 뽑을 때, **"두 모델이 다르면 틀린 것이다"** 라는 전제가 참인가?

이 프로젝트의 설계 전체가 그 전제 위에 서 있다. 참이 아니면 검증 계층을 다시 짜야 한다.

---

## 왜 노트북에서 먼저 하는가

`md-rag-chatbot` 에서 배운 것 — **코드에 상수를 박기 전에 재야 한다.**
거기서 임계값 `0.6` 과 `task_type` 분기는 "그럴듯해서" 들어간 숫자였고, 재보니 둘 다 틀렸다.

노트북은 **틀린 경로도 남긴다.** 최종 답만 본 코드로 넘어간다.

---

## 재는 대상

| | 모델 | 벤더 | 성격 |
|---|---|---|---|
| A | `gemini-2.5-flash` | Google | 상용 폐쇄 가중치 |
| B | `qwen/qwen3.8-27b` | Alibaba 가중치 · Groq 서빙 | 오픈소스 가중치 |

계열이 달라야 **같은 실수를 할 확률이 낮다.** 같은 모델의 크기만 다른 두 개로 갔다면
교차검증의 전제부터 위태로웠다.

---

## 단계

| | 무엇을 | 왜 |
|---|---|---|
| **Phase 0** | 데이터 로드 + **지표 정의** | `md-rag-chatbot` 에서 가장 많이 틀린 자리가 지표였다 |
| Phase 1 | 두 모델 기준선 정확도 | 무엇이 좋은지 알아야 다음이 있다 |
| **Phase 2** | **교차검증 전제 검증** | 설계의 전제. 여기가 이 노트북의 목적 |
| Phase 3 | 코드 검산 검출력 | CORD · 영양성분표 필요 |
| Phase 4 | 자동/보류 판정 기준 | `md-rag-chatbot` 의 `RERANK_THRESHOLD` 에 해당 |

**오늘은 Phase 0 까지.**

---

# Phase 0 — 지표부터 정한다

**먼저 숫자를 뽑고 싶은 유혹을 참는 자리다.**

`md-rag-chatbot` 실험① 에서 *분리도(평균의 차이)* 로 재다가 "검색이 나쁘다" 는 오판을 했다.
평균은 분포의 겹침을 못 본다. 지표가 틀리면 결론도 틀린다 — 그때는 그걸 몰랐다.

여기서 미리 정해야 할 것 세 가지.

### ① 필드 하나가 "맞았다" 는 무엇인가

```
정답      ₩1,000
모델 A    1000
모델 B    1,000원
```

셋 다 같은 값인데 문자열로는 전부 다르다. **완전일치로 재면 정확도가 실제보다 낮게 나온다.**
→ 필드 종류별로 **정규화 후 비교**한다. 금액은 숫자로, 날짜는 ISO 로, 상호명은 공백·대소문자 정리.

**★ 그런데 정규화를 세게 하면 진짜 오류까지 덮는다.** 이 경계를 어디에 둘지가 Phase 0 의 숙제다.

### ② 필드마다 자가 다른데 어떻게 합치나

금액은 숫자 비교, 상호명은 문자열 비교다. 단순 평균을 내면 **어느 필드가 중요한지가 사라진다.**
→ 필드별 정확도를 따로 보고, 합칠 때는 가중을 명시한다.

### ③ "교차검증이 잘 작동한다" 를 하나의 숫자로

이게 제일 어렵다. Phase 2 에서 나올 네 칸이 답을 정한다.

```
                    실제로 맞음      실제로 틀림
두 모델 일치          ✅ 자동처리       ❌ 못 잡는다  ← 이 비율이 설계의 상한
두 모델 불일치        ⚠️ 헛경보         ✅ 잡았다
```

**우하단이 크고 우상단이 작으면** 교차검증이 쓸모 있다.
**우상단이 크면** 교차검증만으로는 부족하고 코드 검산이 더 중요해진다.

→ 재기 전에는 모른다. 그래서 이 노트북이 있다.

In [1]:
import os, sys, json, time, base64, io, re
from pathlib import Path

# ★ 출력 정리 — 원인을 알고 끄는 것이므로 남긴다 (md-rag-chatbot 에서 같은 처리를 했다)
import warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore", message=".*IProgress.*")

DATA = Path("data"); DATA.mkdir(exist_ok=True)
CACHE = DATA / "calls.pkl"          # 모델 호출 결과 캐시 — Groq 가 4장/분이라 필수

# ★ 키를 찾는 순서 — 환경변수 → 이 저장소의 .env → EXTRA_ENV_FILE 로 지정한 파일
#   개인 경로를 코드에 박지 않는다. 다른 저장소의 .env 를 재사용하려면 노트북을 띄울 때
#   EXTRA_ENV_FILE=~/어딘가/.env jupyter lab  처럼 넘긴다.
def env(name: str) -> str:
    if v := os.environ.get(name):
        return v
    paths = [Path("../backend/.env"), Path("../.env")]
    if extra := os.environ.get("EXTRA_ENV_FILE"):
        paths.append(Path(extra).expanduser())
    for path in paths:
        if path.exists() and (m := re.search(rf"^{name}=(.+)$", path.read_text(), re.M)):
            return m.group(1).strip().strip('"\'')
    raise RuntimeError(f"{name} 를 찾을 수 없습니다. backend/.env 에 넣으세요.")

GEMINI_KEY, GROQ_KEY = env("GEMINI_API_KEY"), env("GROQ_API_KEY")
MODEL_A, MODEL_B = "gemini-2.5-flash", "qwen/qwen3.8-27b"
print(f"모델 A  {MODEL_A}\n모델 B  {MODEL_B}")
print(f"키      Gemini {'있음' if GEMINI_KEY else '없음'} · Groq {'있음' if GROQ_KEY else '없음'}")

모델 A  gemini-2.5-flash
모델 B  qwen/qwen3.8-27b
키      Gemini 있음 · Groq 있음


## 0-1. 한도 — 실험 규모를 정하는 값

오늘 실측한 것. **Groq 의 분당 토큰이 병목이다.**

| | 이미지 1장 | 한도 | 처리량 |
|---|---|---|---|
| Gemini | 470 토큰 | 1,500 요청/일 | 여유 |
| **Groq** | **1,873 토큰** | **8,000 토큰/분** | **약 4장/분** |

200장 교차검증이면 **약 50분.** 그래서 **호출 결과를 반드시 캐시**한다.
`md-rag-chatbot` 의 `embed_cache.pkl` 과 같은 이유 — 노트북을 다시 돌릴 때마다 50분을 쓸 수 없다.

In [2]:
# 한도를 헤더로 다시 확인 — 계정 상태가 바뀌었을 수 있다
import httpx
r = httpx.post("https://api.groq.com/openai/v1/chat/completions",
               headers={"Authorization": f"Bearer {GROQ_KEY}"},
               json={"model": MODEL_B, "max_tokens": 5,
                     "messages": [{"role": "user", "content": "hi"}]}, timeout=30)
lim = {k: v for k, v in r.headers.items() if "ratelimit" in k.lower()}
for k, v in sorted(lim.items()):
    print(f"  {k:34} {v}")

tpm = int(lim.get("x-ratelimit-limit-tokens", 0))
print(f"\n이미지 1장 ≈ 1,873 토큰 → 분당 {tpm // 1873 if tpm else '?'}장")
print(f"200장 교차검증 ≈ {200 / (tpm/1873):.0f}분" if tpm else "")

  x-ratelimit-limit-requests         1000
  x-ratelimit-limit-tokens           8000
  x-ratelimit-remaining-requests     998
  x-ratelimit-remaining-tokens       7982
  x-ratelimit-reset-requests         2m52.8s
  x-ratelimit-reset-tokens           134ms

이미지 1장 ≈ 1,873 토큰 → 분당 4장
200장 교차검증 ≈ 47분


## 0-2. 데이터 — SROIE

> ### ⚠️ 2026-09-07 정정 — 이 절의 결론은 뒤집혔다
> 아래 표의 라이선스 두 칸이 **둘 다 틀렸다.** 지우지 않고 남긴다 — 어떻게 틀렸는지가
> 이 노트북의 결과물이다. 정정된 판단은 **`0-2b`** 에 있다.

**정답 라벨이 있고 상업적 사용이 가능한** 유일한 선택지다.

| 데이터셋 | 라이선스 | 판단 |
|---|---|---|
| ~~**SROIE**~~ | ~~**MIT**~~ → **미표기** | ~~채택~~ → **제외** |
| ~~CORD~~ | ~~CC BY-SA 4.0~~ → **CC BY 4.0** | ~~Phase 3 에서 검토~~ → **채택** |
| FUNSD | 비상업 학술 전용 | **제외** — 이 저장소는 공개물이라 충돌 여지 |

### ★ 설계를 세운 뒤에 발견한 것

SROIE 의 필드는 **`company` · `address` · `total` · `date` 넷뿐이고 품목 목록이 없다.**

```
검산 ① 칼로리 = 탄단지 환산    영양성분표에만
검산 ② 1회 제공량 × 횟수       영양성분표에만
검산 ③ 부분 ≤ 전체             품목 필요
       합계 = 품목합           SROIE 엔 품목이 없어 불가
```

**즉 SROIE 로는 "얼마나 정확히 읽나" 는 재지만 "검산이 오류를 잡나" 는 못 잰다.**
검증 계층이 이 프로젝트의 핵심인데 그걸 잴 데이터가 없다.

→ Phase 3 은 CORD 또는 직접 촬영한 영양성분표가 필요하다. ~~**Phase 0~2 는 SROIE 로 충분하다.**~~

★ 이 지적은 여전히 옳다. 그리고 `0-2b` 의 라이선스 대조가 **같은 방향을 가리켰다** —
두 개의 독립적인 이유가 CORD 를 지목한 셈이다.

---

## 0-2b. ★ 정정 — 라이선스를 원본에서 대조했더니 순위가 뒤집혔다

바로 아래 셀에 *"미러를 쓸 때는 라이선스 표기를 원본과 대조한다"* 고 써뒀다.
**대조해보니 우리가 적어둔 `MIT` 자체가 그 함정이었다.**

### MIT 는 어디서 온 것인가

`github.com/zzzDavid/ICDAR-2019-SROIE` 에 MIT LICENSE 파일이 있다. 그런데:

| | |
|---|---|
| MIT 가 덮는 것 | **그 저장소의 코드** — "this software and associated files" |
| 그 저장소에 데이터가 있나 | **없다.** 구글드라이브/Baidu 링크 안내 + 라벨 오류 수정 스크립트뿐 |
| 데이터셋 라이선스 언급 | **없다** |

즉 **대회 참가팀 코드의 라이선스를 데이터셋 라이선스로 읽었다.**

### 원본 확인 — 어디에도 표기가 없다

| 출처 | 라이선스 표기 |
|---|---|
| 공식 RRC 페이지 `rrc.cvc.uab.es/?ch=13` | **없음** (다운로드는 포털 로그인 뒤) |
| 대회 논문 arXiv 2103.10213 | **없음** — CC BY 4.0 은 **논문 문서**에 붙은 것 |
| HF `rth/sroie-2019-v2` · `darentang/sroie` · `Metric-AI/icdar_sroie` | **전부 미표기** |
| HF `jsdnrs/ICDAR2019-SROIE` | CC-BY-4.0 — 업로더 자기 선언, 원본 근거 없음 |

**SROIE 는 금지된 게 아니라 "알 수 없는" 데이터셋이다.** FUNSD 를 뺀 기준을 그대로 적용하면
SROIE 는 애초에 그 심사대에 오른 적조차 없다 — 근거 문서를 본 적이 없기 때문이다.

### 그리고 CORD 는 우리가 적은 것보다 조건이 가볍다

| | 우리가 적어둔 것 | 실제 |
|---|---|---|
| CORD | CC BY-**SA** 4.0 (출처표기 + 동일조건) | **CC BY 4.0 — 출처표기만** |

원본 `clovaai/cord` README 첫 줄 배지와 HF `naver-clova-ix/cord-v2` 메타데이터가 **일치한다.**
SA 를 잘못 붙여서 CORD 를 실제보다 무거운 선택지로 취급하고 있었다.

### 결론 — CORD 단독

|  | 라이선스 근거 | 품목(Phase 3) | 필드 |
|---|---|---|---|
| **CORD** | **명확 · 원본↔미러 일치** | **있다** | 4범주 30종 |
| SROIE | 없음 | 없다 | 4개 |

**능력이 포함관계다.** CORD 에서 `total`·`date` 만 뽑으면 SROIE 수준의 단순 실험이 그대로 되고,
반대는 안 된다. 게다가 CORD 를 쓰면 Phase 3 이 **영양성분표 촬영을 기다리지 않고** 시작된다.

> ★ 배운 것: 라이선스는 **미러가 아니라 원본에서** 확인한다. 그리고 저장소에 붙은 라이선스가
> 덮는 대상이 **코드인지 데이터인지** 구분한다. Groq 의 `403` 을 모델 탓으로 읽을 뻔한 것과
> 같은 종류의 오독이다 — **표기를 봤지 대상을 안 봤다.**

### 덤 — 지표 설계가 오히려 제대로 된다

`0-1` 의 숙제 ②(*필드마다 성격이 다른데 어떻게 합치나*)는 SROIE 4필드에선 잘 드러나지 않는다.
CORD 의 30종 엔티티에서는 피할 수 없다. 본 코드가 **임의 스키마를 받는 엔진**인 이상
어려운 쪽에서 지표를 정하는 게 맞다.

In [ ]:
# CORD 내려받기 — 출처·라이선스를 코드에 남긴다
#
# CORD: A Consolidated Receipt Dataset for Post-OCR Parsing
#   Park, Shin, Lee, Lee, Surh, Seo, Lee (NAVER CLOVA) · DI Workshop @ NeurIPS 2019
#   원본 https://github.com/clovaai/cord
#   배포 https://huggingface.co/datasets/naver-clova-ix/cord-v2
#   라이선스 CC BY 4.0 — 출처표기 + 변경사항 명시 (README "데이터" 절에 적었다)
#
# ★ 라이선스는 원본에서 확인했다. 미러의 표기를 믿지 않는다 (0-2b 참고)
#
# 아직 실행하지 않는다 — 다음 단계에서 한 블록씩 짠다.
print("CORD 로드 — 다음 단계")

---

# 2026-09-07 — 데이터셋 확정

- 라이선스를 **원본에서** 대조 → `SROIE = MIT` 이 틀렸음을 확인 (`0-2b`)
- **CORD 단독 채택.** CC BY 4.0 · 품목 있음 · SROIE 능력을 포함한다
- README 에 CC BY 4.0 인용 블록(저작자·출처·라이선스·변경여부) 추가

---

# 2026-09-06 — 여기까지

**한 일**
- 모델 두 개 확정 (`gemini-2.5-flash` · `qwen/qwen3.8-27b`) — 비전 지원을 **실측으로** 확인
- 한도 실측 — Groq 분당 토큰이 병목, 약 4장/분
- 지표 설계의 숙제 세 가지를 적어둠
- **SROIE 로는 검산을 못 잰다**는 것을 발견 (설계 후에)

**다음**
1. ~~SROIE 내려받기 + 라이선스 대조~~ → **끝. 대조 결과 SROIE 를 버리고 CORD 로 갔다 (`0-2b`)**
2. **CORD 내려받기** + 라벨 구조 파악 (4범주 30종)
3. Phase 0 지표 확정 — 정규화를 어디까지 할 것인가
4. Phase 1 두 모델 기준선
5. Phase 2 교차검증 전제 검증 ← 이 노트북의 목적

**사람이 해야 할 것**
- 영양성분표 **20~30장 촬영**. ★ CORD 로 바꾸면서 **Phase 3 을 막지 않게 됐다** —
  검산 검출력은 CORD 로 먼저 재고, 촬영분은 "직접 만든 평가셋" 축으로 나중에 붙인다.